# 05 — Stage 1b: Enumeration YOLO (deteksi gigi → FDI)

Detektor gigi 32-kelas FDI (kuadran+nomor). Dipakai untuk **assign FDI** ke deteksi penyakit → Stage 1 = **bbox + FDI + diagnosis** (prediksi penuh, tanpa GT).

Data: DENTEX enumeration (634 OPG, 18,095 gigi). Runtime: **GPU**. Output: `yolov8_enum.pt` di Drive.

## Cell 1 — Mount + sync

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/opg-live'
import os
REPO = '/content/opg-live'
if not os.path.exists(REPO):
    !git clone https://github.com/AndreasTopuh/opg-live.git {REPO}
!cd {REPO} && git fetch origin && git reset --hard origin/main && git log --oneline -1

## Cell 2 — Install + convert enumeration → YOLO (32 kelas FDI)

In [ ]:
%pip install -q ultralytics
!python /content/opg-live/scripts/dentex_to_yolo_enum.py --drive {DRIVE_ROOT} --out /content/yolo_enum
!head -3 $(ls /content/yolo_enum/labels/train/*.txt | head -1); echo '---'; head -8 /content/yolo_enum/dentex_enum.yaml

## Cell 3 — Train YOLOv8m (32 kelas gigi)
Gigi besar & teratur → lebih mudah dari penyakit, biasanya mAP tinggi. `batch=16` (T4).

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8m.pt')
model.train(
    data='/content/yolo_enum/dentex_enum.yaml',
    epochs=60, imgsz=1024, batch=16,
    project='/content/runs', name='enum', exist_ok=True, patience=15,
)

## Cell 4 — Simpan ke Drive + eval

In [ ]:
import shutil
shutil.copy('/content/runs/enum/weights/best.pt', f'{DRIVE_ROOT}/checkpoints/yolov8_enum.pt')
print('✅ saved', f'{DRIVE_ROOT}/checkpoints/yolov8_enum.pt')
from ultralytics import YOLO
m = YOLO(f'{DRIVE_ROOT}/checkpoints/yolov8_enum.pt').val(data='/content/yolo_enum/dentex_enum.yaml', imgsz=1024)
print('mAP50:', round(float(m.box.map50), 4), '| mAP50-95:', round(float(m.box.map), 4))

## Cell 5 — Preview: deteksi gigi + FDI

In [ ]:
import glob, matplotlib.pyplot as plt, cv2
from ultralytics import YOLO
model = YOLO(f'{DRIVE_ROOT}/checkpoints/yolov8_enum.pt')
img = sorted(glob.glob('/content/yolo_enum/images/val/*.png'))[0]
res = model.predict(img, imgsz=1024, conf=0.3)[0]
plt.figure(figsize=(18, 8))
plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

## ✅ Deliverable
- [ ] `yolov8_enum.pt` di Drive
- [ ] mAP gigi (biasanya tinggi, gigi teratur)
- [ ] preview: tiap gigi ter-label FDI (11..48)

**Next:** re-run `02_make_artifacts` dengan `--enum_ckpt {DRIVE}/checkpoints/yolov8_enum.pt` → manifest dapat `pred_fdi`. Dan `make_overview --enum_ckpt ...` → overview tampil **FDI + diagnosis** (Stage 1 penuh).